In [1]:
import numpy as np
import matplotlib.pyplot as plt
#from torchvision import datasets, transforms, utils
#from sklearn.ensemble import RandomForestClassifier
#from sklearn.model_selection import ParameterSampler, RandomizedSearchCV, cross_val_score
#from scipy.stats import uniform
from retriever import *
import random
np.random.seed(32)
random.seed(32)
import skopt
from skopt import gp_minimize
from sklearn.model_selection import ParameterSampler, cross_val_score

In [2]:
#Test for request
print(request_location(8,8))

{'results': [{'elevation': 61.34899139404297, 'location': {'lat': 8, 'lng': 8}, 'resolution': 9.543951988220215}], 'status': 'OK'}


### Data retrivel

In [3]:
start = (54,58)
end = (8,15)
#area = request_elevation_area(start, end, 50)
area = {(54.0, 8.0): -30, (54.08163265306123, 8.142857142857142): -18, (54.16326530612245, 8.285714285714286): -16, (54.244897959183675, 8.428571428571429): -9.03537368774414, (54.326530612244895, 8.571428571428571): 0, (54.40816326530612, 8.714285714285714): -0.01181178819388151, (54.48979591836735, 8.857142857142858): -1, (54.57142857142857, 9.0): -1, (54.6530612244898, 9.142857142857142): 18.92743492126465, (54.734693877551024, 9.285714285714286): 24, (54.816326530612244, 9.428571428571429): 0, (54.89795918367347, 9.571428571428571): 2.141496658325195, (54.97959183673469, 9.714285714285714): 20.19146347045898, (55.06122448979592, 9.857142857142858): -3.238455533981323, (55.142857142857146, 10.0): 1.234527230262756, (55.224489795918366, 10.142857142857142): 41.74970245361328, (55.30612244897959, 10.285714285714285): 18.06439971923828, (55.38775510204081, 10.428571428571429): 15.73514747619629, (55.46938775510204, 10.571428571428571): 40.82516479492188, (55.55102040816327, 10.714285714285715): -1.605897307395935, (55.63265306122449, 10.857142857142858): -15.53832244873047, (55.714285714285715, 11.0): 0.002998791169375181, (55.795918367346935, 11.142857142857142): -12.63070487976074, (55.87755102040816, 11.285714285714285): -14, (55.95918367346939, 11.428571428571429): 8.42130184173584, (56.04081632653061, 11.571428571428571): -18.83854866027832, (56.12244897959184, 11.714285714285715): -21.01815414428711, (56.20408163265306, 11.857142857142858): -24.26643371582031, (56.285714285714285, 12.0): -31, (56.36734693877551, 12.142857142857142): -34.27528381347656, (56.44897959183673, 12.285714285714285): -29.50276184082031, (56.53061224489796, 12.428571428571429): -30, (56.61224489795919, 12.571428571428571): -22, (56.69387755102041, 12.714285714285715): 25.6071605682373, (56.775510204081634, 12.857142857142858): 39.28334045410156, (56.857142857142854, 13.0): 134.6513366699219, (56.93877551020408, 13.142857142857142): 133.4641418457031, (57.02040816326531, 13.285714285714285): 152.8367614746094, (57.10204081632653, 13.428571428571429): 181.2979583740234, (57.183673469387756, 13.571428571428571): 150.7835845947266, (57.265306122448976, 13.714285714285715): 175.9743957519531, (57.3469387755102, 13.857142857142858): 279.8223266601562, (57.42857142857143, 14.0): 245.3598022460938, (57.51020408163265, 14.142857142857142): 202.6801910400391, (57.59183673469388, 14.285714285714285): 207.7731628417969, (57.673469387755105, 14.428571428571429): 275.8136596679688, (57.755102040816325, 14.571428571428571): 280.1349487304688, (57.83673469387755, 14.714285714285715): 312.3470764160156, (57.91836734693877, 14.857142857142858): 260.7734375, (58.0, 15.0): 248.5481414794922}

### Bayesian optimization

In [11]:
#Setup

altitude_range = (12.505247, 12.615981)
latitude_range = (55.776345,55.820256)

domain = {
    "latitude": latitude_range,
    "altitude": altitude_range,
}

x0 = [np.random.choice(latitude_range),np.random.choice(altitude_range)]
y0 = request_elevation(x0[0],x0[1])

In [12]:
global i
i = 1
# What we want to maximize
def objective_function(x): 
    
    elevation = request_elevation(x[0],x[1])
    global i
    i += 1
    print(f"\n\n index = {i}, x = {x}, elevation = {elevation}")

    
    return - elevation


np.int = int #numpy np.int deprecation workaround

opt = gp_minimize(objective_function, # the function to minimize
                  [latitude_range, altitude_range],
                                      # the bounds on each dimension of x
                  acq_func="EI",      # the acquisition function
                  n_initial_points=0, # no initial point except the one
                  n_calls=10,         # the number of evaluations of objective_function 19 since we give one initial point
                  x0=x0,           #initial point
                  y0=[y0],           #inital objective function value
                  xi=0.1,             #exploration parameter
                  noise=0.01**2)       # the noise level (optional)



 index = 2, x = [55.820256, 12.505247], elevation = 31.32289886474609


 index = 3, x = [55.78971522966107, 12.506497847028163], elevation = 27.48048973083496


 index = 4, x = [55.820256, 12.530739844556331], elevation = 26.89745330810547


 index = 5, x = [55.808475407712756, 12.556127840281667], elevation = 10.22586154937744


/Users/keremozemre/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/active_machine_learning/week1/02463 Active machine learning and agency, Spring 2026 - 232026 - 157 PM/Active-Machine-Learning-Elevation/.venv/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [55.820256, 12.505247] before, using random point [55.808475407712756, 12.556127840281667]
  warnings.warn(




 index = 6, x = [55.820256, 12.51318746890648], elevation = 30.45187950134277


 index = 7, x = [55.776345, 12.527584658919956], elevation = 38.62041473388672


 index = 8, x = [55.776345, 12.505247], elevation = 31.12199783325195


 index = 9, x = [55.776345, 12.524472537353732], elevation = 32.99466705322266


 index = 10, x = [55.776345, 12.532709301633574], elevation = 43.1845703125


 index = 11, x = [55.787369956884, 12.533772746009634], elevation = 36.52667617797852


In [13]:
y_bo = np.maximum.accumulate(-opt.func_vals).ravel()
y_bo

array([ 1.43106496, 31.32289886, 31.32289886, 31.32289886, 31.32289886,
       31.32289886, 38.62041473, 38.62041473, 38.62041473, 43.18457031,
       43.18457031])

In [14]:
-opt.func_vals.ravel()

array([ 1.43106496, 31.32289886, 27.48048973, 26.89745331, 10.22586155,
       30.4518795 , 38.62041473, 31.12199783, 32.99466705, 43.18457031,
       36.52667618])

In [15]:
param_list = list(ParameterSampler(domain,n_iter=28,random_state=32))
print(param_list)

[{'latitude': 55.776345, 'altitude': 12.505247}, {'latitude': 55.820256, 'altitude': 12.505247}, {'latitude': 55.776345, 'altitude': 12.615981}, {'latitude': 55.820256, 'altitude': 12.615981}]


/Users/keremozemre/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/active_machine_learning/week1/02463 Active machine learning and agency, Spring 2026 - 232026 - 157 PM/Active-Machine-Learning-Elevation/.venv/lib/python3.12/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=28. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
